In [1]:
import math
import dataclasses
import functools
import numpy
from hypothesis import given, strategies as st

In [2]:
def dbg(value):
    print(value)
    return value

In [3]:
def to_bin(frequency: int, resolution: float) -> range:
    lower_bound = int(math.floor((float(frequency) - resolution / 2) / resolution))
    upper_bound = int(math.ceil((float(frequency) + resolution / 2) / resolution))
    return range(lower_bound, (upper_bound + 1))

def test_to_bin():
    assert(to_bin(2400, 300) == range(7, 10))

test_to_bin()

In [4]:
@dataclasses.dataclass(frozen=True)
class Spec:
    sample_rate: int
    mark_frequency: int
    space_frequency: int

    @classmethod
    def with_kcs(cls) -> "Spec":
        return cls(9600, 2400, 1200)

In [5]:
def bins(spec: Spec, window_size: int) -> frozenset[int]:
    f_step = spec.sample_rate / window_size
    mark_bin = to_bin(spec.mark_frequency, f_step)
    space_bin = to_bin(spec.space_frequency, f_step)

    b = set(mark_bin)
    b = b.union(space_bin)
    return frozenset(b)

def test_bins():
    spec = Spec.with_kcs()
    window_size = 32
    b = bins(spec, window_size)
    assert(b == {3, 4, 5, 7, 8, 9})

test_bins()

In [6]:
@dataclasses.dataclass(frozen=True)
class DftTerm:
    frequency: int
    w_real: float
    w_imag: float
    d1: float
    d2: float

    def power(self) -> float:
        return math.pow(self.d2, 2) + math.pow(self.d1, 2) - self.w_real * self.d1 * self.d2

In [7]:
@dataclasses.dataclass(frozen=True)
class Output:
    mark: list[DftTerm]
    space: list[DftTerm]
    max_power: float

In [13]:
def goertzel(spec: Spec, data: list[int]) -> Output:
    window_size = len(data)
    f_step_normalized = 1 / window_size
    sample_rate = float(spec.sample_rate)

    def func(state: (float, float), sample: int) -> (float, float):
        (d1, d2) = state
        y = sample + w_real * d1 - d2
        return (y, d1)

    binned_output = list()
    for k in bins(spec, window_size):
        f = k * f_step_normalized
        wt = 2 * math.pi * f
        w_real = 2 * math.cos(wt)
        w_imag = math.sin(wt)

        (d1, d2) = functools.reduce(func, data, (0, 0))

        binned_output.append(DftTerm(
            int(f * spec.sample_rate),
            w_real,
            w_imag,
            d1,
            d2,
        ))

    max_power = max(binned_output, key=lambda dftt: dftt.power()).power()

    midpoint = (spec.mark_frequency + spec.space_frequency) // 2
    upper_terms = list(filter(lambda dftt: dftt.frequency > midpoint, binned_output))
    lower_terms = list(filter(lambda dftt: dftt.frequency <= midpoint, binned_output))

    if spec.mark_frequency > midpoint:
        return Output(upper_terms, lower_terms, max_power)
    else:
        return Output(lower_terms, upper_terms, max_power)


def test_goertzel_power_2287hz():
    norm_power_threshold = 0.25
    spec = Spec.with_kcs()
    window_size = 32
    freq = 2287
    i8_max = 255
    t = numpy.linspace(0, 1, spec.sample_rate)[:window_size]
    data = (i8_max * numpy.sin(2 * numpy.pi * freq * t))
    output = goertzel(spec, data)
    mark_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.mark)) / len(output.mark)
    space_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.space)) / len(output.space)
    
    assert mark_avg_norm_power > norm_power_threshold, f"{mark_avg_norm_power=}"
    assert space_avg_norm_power < norm_power_threshold, f"{space_avg_norm_power=}"


@given(st.integers(900, 1500))
def test_goertzel_power_space(freq: int):
    norm_power_threshold = 0.25
    spec = Spec.with_kcs()
    window_size = 32
    i8_max = 255
    t = numpy.linspace(0, 1, spec.sample_rate)[:window_size]
    data = (i8_max * numpy.sin(2 * numpy.pi * freq * t))
    output = goertzel(spec, data)
    mark_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.mark)) / len(output.mark)
    space_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.space)) / len(output.space)
    assert mark_avg_norm_power < norm_power_threshold, f"{mark_avg_norm_power=}"
    assert space_avg_norm_power > norm_power_threshold, f"{space_avg_norm_power=}"


@given(st.integers(2100, 2700))
def test_goertzel_power_mark(freq: int):
    norm_power_threshold = 0.25
    spec = Spec.with_kcs()
    window_size = 32
    i8_max = 255
    t = numpy.linspace(0, 1, spec.sample_rate)[:window_size]
    data = (i8_max * numpy.sin(2 * numpy.pi * freq * t))
    output = goertzel(spec, data)
    mark_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.mark)) / len(output.mark)
    space_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.space)) / len(output.space)
    assert mark_avg_norm_power > norm_power_threshold, f"{mark_avg_norm_power=}"
    assert space_avg_norm_power < norm_power_threshold, f"{space_avg_norm_power=}"


@given(st.one_of(st.integers(1, 899), st.integers(1501, 2099), st.integers(2701, 4800)))
def test_goertzel_power_out_of_bounds(freq: int):
    norm_power_threshold = 0.25
    spec = Spec.with_kcs()
    window_size = 32
    i8_max = 255
    t = numpy.linspace(0, 1, spec.sample_rate)[:window_size]
    data = (i8_max * numpy.sin(2 * numpy.pi * freq * t))
    output = goertzel(spec, data)
    mark_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.mark)) / len(output.mark)
    space_avg_norm_power = sum(map(lambda dftt: dftt.power() / output.max_power, output.space)) / len(output.space)
    assert mark_avg_norm_power < norm_power_threshold, f"{mark_avg_norm_power=} !< {norm_power_threshold=}"
    assert space_avg_norm_power < norm_power_threshold, f"{space_avg_norm_power=} !< {norm_power_threshold=}"


test_goertzel_power_2287hz()
test_goertzel_power_space()
test_goertzel_power_mark()
test_goertzel_power_out_of_bounds()

  + Exception Group Traceback (most recent call last):
  |   File "/nix/store/926nzckk77h55v4q92i9hx0w559zjbr4-python3-3.13.11-env/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3699, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/tmp/nix-shell.6gi0rM/ipykernel_67549/3325921860.py", line 104, in <module>
  |     test_goertzel_power_out_of_bounds()
  |     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  |   File "/tmp/nix-shell.6gi0rM/ipykernel_67549/3325921860.py", line 87, in test_goertzel_power_out_of_bounds
  |     def test_goertzel_power_out_of_bounds(freq: int):
  |                    ^^^
  |   File "/nix/store/926nzckk77h55v4q92i9hx0w559zjbr4-python3-3.13.11-env/lib/python3.13/site-packages/hypothesis/core.py", line 2110, in wrapped_test
  |     raise the_error_hypothesis_found
  | ExceptionGroup: Hypothesis found 2 distinct failures. (2 sub-exceptions)
  +-+---------------- 